# 01 · Data Understanding (CRISP-DM step 2)

**Goal:** get to know the USK-Coffee images before touching any model.

We check: how many images per class and split, whether every file opens, image sizes
and color modes, what each class looks like, and whether classes differ in color or
brightness (which tells us what the CNN might rely on).

**Outputs:** `02_data/interim/inventory.csv`, `02_data/interim/color_stats.csv`,
and figures in `reports/figures/` (prefix `01_`).

*Runs fine on a laptop CPU or on Colab (no GPU needed for this notebook).*

In [ ]:
# --- SETUP: run this cell first -----------------------------------------
import os, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    # Change this if you uploaded the project folder somewhere else in your Drive:
    PROJECT = Path("/content/drive/MyDrive/kapeklasipika")
    %pip install -q imagehash
else:
    # Running on your laptop: walk up from this notebook until we find config/config.yaml
    PROJECT = Path.cwd()
    while not (PROJECT / "config" / "config.yaml").exists():
        PROJECT = PROJECT.parent

os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))

from src.config import load_config, use_local_copy
cfg = load_config()
print("Project folder:", PROJECT)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.data_loading import build_inventory, CLASS_NAMES, SPLITS
from src.preprocessing import image_color_stats
from src.evaluate import show_image_grid

FIG = cfg["paths"]["figures"]; FIG.mkdir(parents=True, exist_ok=True)
INTERIM = cfg["paths"]["interim"]; INTERIM.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid")

## 1. Inventory: one row per image
`build_inventory` walks through every `split/class` folder and records the file size,
image width and height, color mode, and whether the file can be opened.

In [ ]:
inv = build_inventory(cfg["paths"]["raw"])
inv.to_csv(INTERIM / "inventory.csv", index=False)
print(f"{len(inv):,} images found")
inv.head()

## 2. Class balance per split
A balanced dataset means accuracy is a fair headline metric and no resampling is needed.

In [ ]:
counts = pd.crosstab(inv["split"], inv["label"]).reindex(SPLITS)[CLASS_NAMES]
counts["total"] = counts.sum(axis=1)
display(counts)

ax = pd.crosstab(inv["label"], inv["split"])[SPLITS].plot(kind="bar", figsize=(7, 4), rot=0)
ax.set_xlabel("Class"); ax.set_ylabel("Number of images"); ax.set_title("Images per class and split")
plt.tight_layout(); plt.savefig(FIG / "01_class_counts.png", dpi=200); plt.show()

## 3. File quality: unreadable files, sizes, color modes

In [ ]:
print("Unreadable files:", (~inv["readable"]).sum())
print("\nImage sizes (width x height):")
print(inv.groupby(["width", "height"]).size().rename("count"))
print("\nColor modes:")
print(inv["mode"].value_counts())
print("\nFile size (KB):")
inv["file_size_kb"].describe().round(1)

**How to read this:** we expect 0 unreadable files, every image 256 × 256, and mode `RGB`.
Anything else (grayscale `L`, `RGBA`, odd sizes, tiny files) is a data quality issue to record
in `03_exploration/findings.md`.

## 4. What does each class look like?
Six random training images per class.

In [ ]:
sample = inv[inv["split"] == "train"].groupby("label").sample(6, random_state=cfg["seed"])
show_image_grid(sample["file_path"].tolist(), sample["label"].tolist(), ncols=6,
                save_path=FIG / "01_samples_per_class.png")
plt.show()

## 5. Color and brightness per class
If one class is clearly darker or a different color, the CNN may learn color as a shortcut.
That is fine when the color difference is real (e.g. dark defective beans), but it also means
the model may be sensitive to lighting changes in real phone photos.

In [ ]:
stats = image_color_stats(inv)
stats.to_csv(INTERIM / "color_stats.csv", index=False)

display(stats.groupby("label")[["mean_r", "mean_g", "mean_b", "brightness"]].agg(["mean", "std"]).round(1))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.boxplot(data=stats, x="label", y="brightness", order=CLASS_NAMES, ax=axes[0])
axes[0].set_title("Brightness per class"); axes[0].set_xlabel("Class"); axes[0].set_ylabel("Mean brightness (0-255)")
sns.scatterplot(data=stats.sample(min(2000, len(stats)), random_state=0),
                x="mean_r", y="mean_g", hue="label", hue_order=CLASS_NAMES, s=12, alpha=0.6, ax=axes[1])
axes[1].set_title("Average red vs green per image"); axes[1].set_xlabel("Mean red"); axes[1].set_ylabel("Mean green")
plt.tight_layout(); plt.savefig(FIG / "01_color_brightness.png", dpi=200); plt.show()

## 6. Privacy check
The dataset contains only photos of coffee beans: no people, faces, names, or IDs.
No privacy handling is needed. (Your own phone photos should also show beans only.)

## 7. Next step
Copy the printed tables and look at the figures, then write the main observations in
`03_exploration/findings.md` (or paste the outputs to Claude and it will draft them).